In [188]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [189]:
from google.cloud import bigquery
from textblob import TextBlob
import pandas as pd

In [190]:
# Initialize BigQuery client
bq_client = bigquery.Client()


Using Kaggle's public dataset BigQuery integration.


In [191]:
# SAMPLE COMMENTS FIRST 
query_comments = """
    SELECT id, text, post_id, user_id, score
    FROM `bigquery-public-data.stackoverflow.comments`
    WHERE RAND() < 0.05
    LIMIT 110000
"""
comments_df = bq_client.query(query_comments).to_dataframe()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [192]:
print(f"Sampled comments: {comments_df.shape}")

Sampled comments: (110000, 5)


In [193]:
# GET UNIQUE POST_IDS AND USER_IDS
post_ids = comments_df["post_id"].dropna().unique().tolist()
user_ids = comments_df["user_id"].dropna().unique().tolist()

print(f"Unique post IDs: {len(post_ids)}")
print(f"Unique user IDs: {len(user_ids)}")

Unique post IDs: 100818
Unique user IDs: 54836


In [194]:
# GET ONLY MATCHING POSTS
query_posts = f"""
    SELECT id, comment_count, favorite_count, score
    FROM `bigquery-public-data.stackoverflow.posts_answers`
    WHERE id IN UNNEST(@post_ids)
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("post_ids", "INT64", post_ids)
    ]
)
posts_df = bq_client.query(query_posts, job_config=job_config).to_dataframe()
print(f"Fetched posts: {posts_df.shape}")


/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Fetched posts: (50853, 4)


In [195]:
# GET ONLY MATCHING USERS
query_users = f"""
    SELECT id, display_name, age, location, reputation, up_votes, down_votes
    FROM `bigquery-public-data.stackoverflow.users`
    WHERE id IN UNNEST(@user_ids)
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("user_ids", "INT64", user_ids)
    ]
)
users_df = bq_client.query(query_users, job_config=job_config).to_dataframe()
print(f"Fetched users: {users_df.shape}")

Fetched users: (54836, 7)


In [196]:
# SAVE TO CSV
comments_df.to_csv("/kaggle/working/comments.csv", index=False)
posts_df.to_csv("/kaggle/working/posts_answers.csv", index=False)
users_df.to_csv("/kaggle/working/users.csv", index=False)